# Multiple Dataset

In [ ]:
order = [
    #Popular Representation
    'pipe_serialized',
    'token_serialized',
    'space_serialized',
    "centroid_popular",
    # Data Representation
    'csv',
    'tsv',
    'html',
    'markdown',
    'latex',
    'dict',
    'json',
    'xml',
    "centroid_data",
    # Structural Transformations 
    'shuffled_rows',
    'shuffled_cols',
    'transpose',
    "centroid_structural",
    #Schema Definition Types
    'mschema',
    'macschema',
    'ddl',
    "centroid_schema",
     "centroid_all", 
]
category = {"Popular Representation": ['pipe_serialized',    'token_serialized', 'space_serialized',"centroid_popular"],
    "Data Representation" : [
            'csv',
            'tsv',
            'html',
            'markdown',
            'latex',
            'dict',
            'json',
            'xml',
            'centroid_data'],
    "Structural Transformations": ['shuffled_rows',
        'shuffled_cols',
        'transpose',
        'centroid_structural'],
    "Schema Definition Types" :['mschema',
    'macschema',
    'ddl',
    'centroid_schema'],
    "All" :['centroid_all'],
    "Top 5": [  "tsv_csv_space_serialized_ddl_latex","tsv_csv_pipe_serialized_space_serialized_transpose","xml_dict_json_pipe_serialized_markdown","xml_ddl_latex_html_token_serialized"],
    "Top 5 All": ["csv_tsv_pipe_serialized_space_serialized_xml_latex"],
}
category_wo_centroid = {"Popular Representation": ['pipe_serialized',    'token_serialized', 'space_serialized'],
    "Data Representation" : [
            'csv',
            'tsv',
            'html',
            'markdown',
            'latex',
            'dict',
            'json',
            'xml',
            ],
    "Structural Transformations": ['shuffled_rows',
        'shuffled_cols',
        'transpose'],
    "Schema Definition Types" :['mschema',
    'macschema',
    'ddl'],
    "Top 5": [   "tsv_csv_space_serialized_ddl_latex","tsv_csv_pipe_serialized_space_serialized_transpose","xml_dict_json_pipe_serialized_markdown","xml_ddl_latex_html_token_serialized"],
     "Top 5 All": ["csv_tsv_pipe_serialized_space_serialized_xml_latex"]
}

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
recall_value = 1
import matplotlib.pyplot as plt

base_models = ["mpnet", "reasonir", "bge", "splade"]
variants = ["", "_adapter", "_adapter_subset"]

# One colormap per base model (pick any you like, but keep them distinct)
cmaps = {
    "mpnet": plt.cm.Blues,
    "reasonir": plt.cm.PuRd,
    "bge": plt.cm.YlGn,
    "splade": plt.cm.OrRd,
}

# 3 shades per base model (dark -> light)
shade_levels = [0.85, 0.65, 0.45]

# Build hue_order and palette mapping
hue_order = [m + v for m in base_models for v in variants]
palette = {
    (m + v): cmaps[m](shade_levels[i])
    for m in base_models
    for i, v in enumerate(variants)
}

for recall_value in [1]:
    #for dataset in ["WTQ","WIKISQL","NQ"]:
    for dataset in ["WTQ","WIKISQL","NQ"]:
        for date in ["2026-03-10"]:
            df_list = []
            #for model in ["reasonir","splade","mpnet","bge","bm25"]:
            for model in ["mpnet","reasonir","bge","splade"]:#,
                with open(f'./data/retrieval_all/{model}_results/{dataset}/perturbation_results.json') as f:
                    data = json.load(f)
                if model =="splade":
                    date = "2026-03-01"
                with open(f'./data/retrieval_all/{model}_results/{dataset}/{date}/perturbation_results_with_adapter.json') as f:
                    data_adapter = json.load(f)
                with open(f'./data/retrieval_all/{model}_results/{dataset}/{date}/perturbation_results_with_adapter_subset.json') as f:
                    data_adapter_subset = json.load(f)
                df = pd.DataFrame(data).T
                df_adapt = pd.DataFrame(data_adapter).T
                df_adapt_subset = pd.DataFrame(data_adapter_subset).T
                df = df.reset_index().rename(columns={"index": "Index"})
                df_adapt = df_adapt.reset_index().rename(columns={"index": "Index"})
                data_adapter_subset = df_adapt_subset.reset_index().rename(columns={"index": "Index"})
                df["Representation"] = df["Index"].apply(lambda x: x.split("+")[-1].strip()) 
                df_adapt["Representation"] = df_adapt["Index"].apply(lambda x: x.split("+")[-1].strip()) 
                data_adapter_subset["Representation"] = data_adapter_subset["Index"].apply(lambda x: x.split("+")[-1].strip()) 
                df["Model"]= [model]*df.shape[0]
                df_adapt["Model"]= [model+"_adapter"]*df_adapt.shape[0]
                data_adapter_subset["Model"]= [model+"_adapter_subset"]*data_adapter_subset.shape[0]
                df_list+= [df,df_adapt,data_adapter_subset]
            df = pd.concat(df_list)
            df = df.reset_index(drop=True)
            print(df["Model"].unique())
            #print(df[df[[x for x in df.columns if x!="combo_reps"]].duplicated(keep=False)])

            plt.figure(figsize=(24, 6))

            ax = sns.barplot(
                data=df,
                x="Representation",
                y=f"recall@{recall_value}",
                hue="Model",
                hue_order=hue_order,
                order=order,
                palette=palette
            )

            # ---------- category structure ----------
            categories_in_order = [
                "Popular Representation",
                "Data Representation",
                "Structural Transformations",
                "Schema Definition Types",
                "All",
            ]

            category_sizes = [
                len(category["Popular Representation"]),
                len(category["Data Representation"]),
                len(category["Structural Transformations"]),
                len(category["Schema Definition Types"]),
                len(category["All"]),
            ]

            # cumulative positions
            starts = np.cumsum([0] + category_sizes[:-1])
            ends = np.cumsum(category_sizes)
            centers = (starts + ends - 1) / 2

            # ---------- vertical separators ----------
            for boundary in ends[:-1]:
                ax.axvline(
                    boundary - 0.5,
                    color="black",
                    linestyle="-",
                    linewidth=1.2,
                    alpha=0.5
                )

            # ---------- category titles ----------
            ymax = ax.get_ylim()[1]
            for center, label in zip(centers, categories_in_order):
                ax.text(
                    center,
                    ymax * 1.03,      # push above bars
                    label,
                    ha="center",
                    va="bottom",
                    fontsize=11,
                    fontweight="bold"
                )

            # ---------- labels & formatting ----------
            plt.ylabel(f"recall@{recall_value}")
            plt.xlabel("Serialization Method")
            #plt.title(f"{dataset} | recall@{recall_value} by Serialization Method", pad=30)

            plt.xticks(rotation=45, ha="right")

            # ---------- annotate bars ----------
            for p in ax.patches:
                height = p.get_height()
                if not pd.isna(height):
                    ax.annotate(
                        f"{height:.2f}",
                        (p.get_x() + p.get_width() / 2, height),
                        ha="center",
                        va="bottom",
                        fontsize=8,
                        xytext=(0, 3),
                        textcoords="offset points"
                    )

            # ---------- legend ----------
            ax.legend(
                title="Model",
                bbox_to_anchor=(1, 1),
                loc="upper left",
                frameon=False
            )

            # extra top space for category labels
            plt.subplots_adjust(top=0.80)
            plt.tight_layout()
            plt.show()


# Latex

In [ ]:
all_summary = []
subset_order = [ 'pipe_serialized','token_serialized','space_serialized',
                'csv','tsv', 'html','markdown','latex','dict','json','xml',
                'shuffled_rows','shuffled_cols','transpose',
                'mschema','macschema','ddl',
                "centroid_all",
]
date = "2026-03-10"
base_models = ["mpnet", "reasonir", "bge", "splade"]
base_models_w_adapter = ['mpnet','mpnet_adapter','mpnet_adapter_subset',
                          'reasonir','reasonir_adapter','reasonir_adapter_subset',
                            'bge','bge_adapter','bge_adapter_subset',
                              'splade','splade_adapter','splade_adapter_subset']
recall_value = 1
for dataset in ["WTQ", "WIKISQL", "NQ"]:
    df_list = []
    for model in base_models:
        with open(f'./data/retrieval_all/{model}_results/{dataset}/perturbation_results.json') as f:
            data = json.load(f)
        if model =="splade":
            date = "2026-03-01"
        with open(f'./data/retrieval_all/{model}_results/{dataset}/{date}/perturbation_results_with_adapter.json') as f:
            data_adapter = json.load(f)
        with open(f'./data/retrieval_all/{model}_results/{dataset}/{date}/perturbation_results_with_adapter_subset.json') as f:
            data_adapter_subset = json.load(f)
        df = pd.DataFrame(data).T
        df_adapt = pd.DataFrame(data_adapter).T
        df_adapt_subset = pd.DataFrame(data_adapter_subset).T
        df = df.reset_index().rename(columns={"index": "Index"})
        df_adapt = df_adapt.reset_index().rename(columns={"index": "Index"})
        data_adapter_subset = df_adapt_subset.reset_index().rename(columns={"index": "Index"})
        df["Representation"] = df["Index"].apply(lambda x: x.split("+")[-1].strip()) 
        df_adapt["Representation"] = df_adapt["Index"].apply(lambda x: x.split("+")[-1].strip()) 
        data_adapter_subset["Representation"] = data_adapter_subset["Index"].apply(lambda x: x.split("+")[-1].strip()) 
        df["Model"]= [model]*df.shape[0]
        df_adapt["Model"]= [model+"_adapter"]*df_adapt.shape[0]
        data_adapter_subset["Model"]= [model+"_adapter_subset"]*data_adapter_subset.shape[0]
        df_list+= [df,df_adapt,data_adapter_subset]
    df = pd.concat(df_list).reset_index(drop=True)
    df = df[df["Representation"].isin(subset_order)].copy()
    df["Dataset"]= [dataset]*df.shape[0]
    metric = f"recall@{recall_value}"
    summary = (
        df.groupby(["Dataset", "Model", "Representation"])[metric]
            .agg(["mean", "std"])
            .reset_index()
    )

    summary["mean_std"] = summary.apply(
        lambda row: f"{row['mean']:.2f}",
        axis=1
    )

    all_summary.append(summary)

final_summary = pd.concat(all_summary, ignore_index=True)

latex_table = (
    final_summary.pivot(
        index=["Dataset", "Model"],
        columns="Representation",
        values="mean_std"
    )
    .reindex(columns=subset_order)
    .reindex(index=pd.MultiIndex.from_product(
        [["WTQ", "WIKISQL", "NQ"], base_models_w_adapter],
        names=["Dataset", "Model"]
    ))
)

print(
    latex_table.to_latex(
        escape=False,
        na_rep="",
        multirow=True,
        caption=f"Results for recall@{recall_value}",
        label=f"tab:recall_{recall_value}_by_dataset_model",
        column_format="ll" + "c" * len(latex_table.columns)
    )
)

# HeatMap

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

recall_value = 1

base_models = ["mpnet", "reasonir", "bge", "splade"]
variants = ["", "_adapter", "_adapter_subset"]

datasets = ["WTQ", "WIKISQL", "NQ"]
models = ["mpnet","reasonir","bge", "splade"]#"reasonir", "splade", "mpnet", "bge"]


def load_one(model, dataset,date):
    with open(f"./data/retrieval_all/{model}_results/{dataset}/perturbation_results.json") as f:
        data = json.load(f)
    if model =="splade":
        date = "2026-03-01"
    with open(f"./data/retrieval_all/{model}_results/{dataset}/{date}/perturbation_results_with_adapter.json") as f:
        data_adapter = json.load(f)
    #if model =="bge" and dataset == "NQ":
    #    date = ""
    with open(f"./data/retrieval_all/{model}_results/{dataset}/{date}/perturbation_results_with_adapter_subset.json") as f:
        data_adapter_subset = json.load(f)

    df = pd.DataFrame(data).T.reset_index().rename(columns={"index": "Index"})
    df_adapt = pd.DataFrame(data_adapter).T.reset_index().rename(columns={"index": "Index"})
    df_adapt_subset = pd.DataFrame(data_adapter_subset).T.reset_index().rename(columns={"index": "Index"})

    for d in (df, df_adapt, df_adapt_subset):
        d["Representation"] = d["Index"].apply(lambda x: x.split("+")[-1].strip())
        d["Dataset"] = dataset
    df = df[df["Representation"].isin(order)]
    df["Model"] = model
    df_adapt["Model"] = model + "_adapter"
    df_adapt_subset["Model"] = model + "_adapter_subset"

    return pd.concat([df, df_adapt, df_adapt_subset], ignore_index=True)

for date in ["2026-03-10"]:
    # ---------- build the big dataframe ----------
    df_all = []
    for dataset in datasets:
        for model in models:
            df_all.append(load_one(model, dataset,date))

    df = pd.concat(df_all, ignore_index=True)

    metric_col = f"recall@{recall_value}"

    # Ensure numeric
    df[metric_col] = pd.to_numeric(df[metric_col], errors="coerce").astype("float64")

    # BaseModel and Variant
    df["BaseModel"] = (
        df["Model"]
        .str.replace("_adapter_subset", "", regex=False)
        .str.replace("_adapter", "", regex=False)
    )

    df["Variant"] = "base"
    df.loc[df["Model"].str.endswith("_adapter"), "Variant"] = "adapter"
    df.loc[df["Model"].str.endswith("_adapter_subset"), "Variant"] = "adapter_subset"

    # Average in case you have duplicates
    agg = (
        df.groupby(["Dataset", "BaseModel", "Representation", "Variant"], as_index=False)[metric_col]
        .mean()
    )

    # Pivot so base/adapter/subset become columns
    wide = agg.pivot_table(
        index=["Dataset", "BaseModel", "Representation"],
        columns="Variant",
        values=metric_col,
        aggfunc="mean"
    ).reset_index()

    # Compute deltas vs base
    wide["delta_adapter"] = wide["adapter"] - wide["base"]
    wide["delta_adapter_subset"] = wide["adapter_subset"] - wide["base"]


    dataset = "WTQ"  # or loop over datasets
    for dataset in datasets:
        # If you already have an order list for representations, use it
        if "order" not in globals() or order is None:
            order = sorted(wide["Representation"].unique())

        panels = [
            ("base", "Base"),
            ("delta_adapter", "Adapter − Base"),
            ("delta_adapter_subset", "AdapterSubset − Base"),
        ]

        fig, axes = plt.subplots(1, 3, figsize=(26, 7), sharey=True, constrained_layout=True)

        # Use symmetric color range for deltas
        d = wide[wide["Dataset"] == dataset]
        delta_max = np.nanmax(np.abs(d[["delta_adapter", "delta_adapter_subset"]].to_numpy()))
        delta_max = float(delta_max) if np.isfinite(delta_max) else 0.0

        for ax, (col, title) in zip(axes, panels):
            hm = (
                d.pivot_table(index="BaseModel", columns="Representation", values=col, aggfunc="mean")
                .reindex(columns=order)
                .astype("float64")
            )

            if col == "base":
                sns.heatmap(
                    hm, ax=ax, cmap="viridis", vmin=0.0, vmax=1.0,
                    annot=True, fmt=".2f",
                    cbar=(ax is axes[-1])
                )
            else:
                sns.heatmap(
                    hm, ax=ax, cmap="RdBu_r", vmin=-delta_max, vmax=delta_max,
                    center=0.0, annot=True, fmt=".2f",
                    cbar=(ax is axes[-1])
                )

            ax.set_title(f"{dataset} | {title}")
            ax.set_xlabel("Representation")
            ax.set_ylabel("BaseModel" if ax is axes[0] else "")
            ax.tick_params(axis="x", rotation=45)

        plt.show()

# Compare Standard Deviation

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

category = {
    "Popular Representation": ['pipe_serialized', 'token_serialized', 'space_serialized', "centroid_popular"],
    "Data Representation": ['csv','tsv','html','markdown','latex','dict','json','xml','centroid_data'],
    "Structural Transformations": ['shuffled_rows','shuffled_cols','transpose','centroid_structural'],
    "Schema and Definition Types": ['mschema','macschema','ddl','centroid_schema'],
    "All": ['pipe_serialized', 'token_serialized', 'space_serialized','csv','tsv','html','markdown','latex','dict','json','xml','shuffled_rows','shuffled_cols','transpose','mschema','macschema','ddl','centroid_all']
}

category_wo_centroid = {
    "Popular Representation": ['pipe_serialized','token_serialized','space_serialized'],
    "Data Representation": ['csv','tsv','html','markdown','latex','dict','json','xml'],
    "Structural Transformations": ['shuffled_rows','shuffled_cols','transpose'],
    "Schema and Definition Types": ['mschema','macschema','ddl'],
    "All": ['pipe_serialized', 'token_serialized', 'space_serialized','csv','tsv','html','markdown','latex','dict','json','xml','shuffled_rows','shuffled_cols','transpose','mschema','macschema','ddl'],
}
representation = [
    #Popular Representation
    'pipe_serialized',
    'token_serialized',
    'space_serialized',
    "centroid_popular",
    # Data Representation
    'csv',
    'tsv',
    'html',
    'markdown',
    'latex',
    'dict',
    'json',
    'xml',
    "centroid_data",
    # Structural Transformations 
    'shuffled_rows',
    'shuffled_cols',
    'transpose',
    "centroid_structural",
    #Schema and Definition Types
    'mschema',
    'macschema',
    'ddl',
    "centroid_schema",
     "centroid_all",
]
def load_one(model, dataset,date):
    with open(f"./data/retrieval_all/{model}_results/{dataset}/perturbation_results.json") as f:
        data = json.load(f)
    if model =="splade":
        date = "2026-03-01"
    with open(f"./data/retrieval_all/{model}_results/{dataset}/{date}/perturbation_results_with_adapter.json") as f:
        data_adapter = json.load(f)
    #if model =="bge" and dataset == "NQ":
    #    date = ""
    with open(f"./data/retrieval_all/{model}_results/{dataset}/{date}/perturbation_results_with_adapter_subset.json") as f:
        data_adapter_subset = json.load(f)

    df = pd.DataFrame(data).T.reset_index().rename(columns={"index": "Index"})
    df_adapt = pd.DataFrame(data_adapter).T.reset_index().rename(columns={"index": "Index"})
    df_adapt_subset = pd.DataFrame(data_adapter_subset).T.reset_index().rename(columns={"index": "Index"})

    for d in (df, df_adapt, df_adapt_subset):
        d["Representation"] = d["Index"].apply(lambda x: x.split("+")[-1].strip())
        d["Dataset"] = dataset
    df = df[df["Representation"].isin(order)]
    df["Model"] = model
    df_adapt["Model"] = model + "_adapter"
    df_adapt_subset["Model"] = model + "_adapter_subset"
    return pd.concat([df, df_adapt, df_adapt_subset], ignore_index=True)

recall_value = 1

base_models = ["mpnet", "reasonir", "bge", "splade"]
variants = ["", "_adapter", "_adapter_subset"]

datasets = ["WTQ", "WIKISQL", "NQ"]
models = ["mpnet","reasonir","bge","splade"]#"reasonir", "splade", "mpnet", "bge"]


df = []
for date in ["2026-03-10"]:
    # ---------- build the big dataframe ----------
    df_all = []
    for dataset in datasets:
        for model in models:
            df_all.append(load_one(model, dataset,date))
    df_date= pd.concat(df_all, ignore_index=True)
    df_date["Date"] = [date]*df_date.shape[0]
    df.append(df_date)
df = pd.concat(df, ignore_index=True)

def get_category(rep):
    if rep == "centroid_all":
        return "All"
    for cat, reps in category.items():
        if rep in reps and cat != "All":
            return cat
    return None
df["Category"] = df["Representation"].apply(get_category)
metric = "recall@1"
df["IsCentroid"] = df["Representation"].apply(lambda x: True if 'centroid' in x else False)
df_wo_centroid = df[~df["IsCentroid"]].copy()
df_wo_centroid["rank"] = (
    df_wo_centroid.groupby(["Model", "Dataset"])[metric]
      .rank(ascending=False, method="average")
)
df_wo_centroid =df_wo_centroid.sort_values(["Model", "Dataset","rank"])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metric = 'recall@1'

base_models = ["mpnet", "reasonir", "bge", "splade"]
variants = ["", "_adapter", "_adapter_subset"]

# One colormap per base model (pick any you like, but keep them distinct)
cmaps = {
    "mpnet": plt.cm.Blues,
    "reasonir": plt.cm.PuRd,
    "bge": plt.cm.YlGn,
    "splade": plt.cm.OrRd,
}

# 3 shades per base model (dark -> light)
shade_levels = [0.85, 0.65, 0.45]

# Build hue_order and palette mapping
hue_order = [m + v for m in base_models for v in variants]
palette = {
    (m + v): cmaps[m](shade_levels[i])
    for m in base_models
    for i, v in enumerate(variants)
}


for date, df in df_wo_centroid.groupby("Date"):
    print(date)
    # 1) Keep only non-centroid rows
    if 'IsCentroid' in df.columns:
        df = df[df['IsCentroid'] == False].copy()

    if 'combo_reps' in df.columns:
        df = df[df['combo_reps'].isna()].copy()

    # 2) Compute variation per Dataset x Model
    variation = (
        df.groupby(['Dataset', 'Model'])[metric]
          .agg(std='std', min_val='min', max_val='max', mean='mean', n='count')
          .reset_index()
    )

    variation['range'] = variation['max_val'] - variation['min_val']

    #print("Per Dataset x Model variation:")
    #print(variation.sort_values(['Dataset', 'Model']).to_string(index=False))

    # 3) Dataset-level summary
    dataset_summary = (
        variation.groupby('Dataset')[['std', 'range']]
                 .mean()
                 .reset_index()
                 .rename(columns={
                     'std': 'avg_std_across_models',
                     'range': 'avg_range_across_models'
                 })
    )

    #print("\nDataset-level summary:")
    #print(dataset_summary.sort_values('avg_range_across_models', ascending=False).to_string(index=False))

    # Keep only model names that are both in hue_order and present in this slice
    present_models = [m for m in hue_order if m in variation['Model'].unique()]
    plot_colors = [palette[m] for m in present_models]

    # 4) Plot standard deviation
    std_pivot = (
        variation.pivot(index='Dataset', columns='Model', values='std')
                 .reindex(columns=present_models)
    )

    ax = std_pivot.plot(
        kind='bar',
        figsize=(10, 5),
        rot=0,
        color=plot_colors
    )
    ax.set_title(f'Std Dev of {metric} Across Representations')
    ax.set_xlabel('Dataset')
    ax.set_ylabel('Standard Deviation')
    #ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    # 5) Plot range (max - min)
    range_pivot = (
        variation.pivot(index='Dataset', columns='Model', values='range')
                 .reindex(columns=present_models)
    )

    ax = range_pivot.plot(
        kind='bar',
        figsize=(10, 5),
        rot=0,
        color=plot_colors
    )
    ax.set_title(f'Range of {metric} Across Representations (Max - Min)')
    ax.set_xlabel('Dataset')
    ax.set_ylabel('Range')
    ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    # 6) Single-number view
    #print("\nRanking by avg_range_across_models (higher = more variation):")
    #print(dataset_summary.sort_values('avg_range_across_models', ascending=False).to_string(index=False))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metric = 'recall@1'
base_models = ["mpnet", "reasonir", "bge", "splade"]
variants = ["", "_adapter", "_adapter_subset"]

# One colormap per base model (pick any you like, but keep them distinct)
cmaps = {
    "mpnet": plt.cm.Blues,
    "reasonir": plt.cm.PuRd,
    "bge": plt.cm.YlGn,
    "splade": plt.cm.OrRd,
}

# 3 shades per base model (dark -> light)
shade_levels = [0.85, 0.65, 0.45]

# Build hue_order and palette mapping
hue_order = [m + v for m in base_models for v in variants]
palette = {
    (m + v): cmaps[m](shade_levels[i])
    for m in base_models
    for i, v in enumerate(variants)
}
datasets = ["WTQ", "WIKISQL", "NQ"]
# -----------------------------
# 1) Keep only non-centroid rows
# -----------------------------
df = df_wo_centroid.copy()

# If df_wo_centroid already excludes centroid, this is harmless.
if 'IsCentroid' in df.columns:
    df = df[df['IsCentroid'] == False].copy()

# Optional: exclude combo rows if they exist in this df
# (You said this df is "wo centroid", but combos can still appear)
if 'combo_reps' in df.columns:
    df = df[df['combo_reps'].isna()].copy()

# -----------------------------
# 2) Compute variation per Dataset x Model
# -----------------------------
variation = (
    df.groupby(['Dataset', 'Model'])[metric]
      .agg(std='std', min_val='min', max_val='max', mean='mean', n='count')
      .reset_index()
)

variation['range'] = variation['max_val'] - variation['min_val']  # max - min
present_models = [m for m in hue_order if m in variation['Model'].unique()]
plot_colors = [palette[m] for m in present_models]
print("Per Dataset x Model variation:")
print(variation.sort_values(['Dataset', 'Model']).to_string(index=False))

# -----------------------------
# 3) Dataset-level summary
#    (average variation across models)
# -----------------------------
dataset_summary = (
    variation.groupby('Dataset')[['std', 'range']]
             .mean()
             .reset_index()
             .rename(columns={
                 'std': 'avg_std_across_models',
                 'range': 'avg_range_across_models'
             })
)

print("\nDataset-level summary:")
print(dataset_summary.sort_values('avg_range_across_models', ascending=False).to_string(index=False))

# -----------------------------
# 4) Plot standard deviation
# -----------------------------
std_pivot = variation.pivot(index='Dataset', columns='Model', values='std')
std_pivot = std_pivot.reindex(index=datasets,columns=present_models)
ax = std_pivot.plot(kind='bar', figsize=(8, 4), rot=0,color = palette, width=0.85)
#ax.set_title(f'Std Dev of {metric} Across Representations')
#ax.set_xlabel('Dataset')
#ax.set_ylabel('Standard Deviation')
#ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.get_legend().remove()
plt.tight_layout()
plt.show()

# -----------------------------
# 5) Plot range (max - min)
# -----------------------------
range_pivot = variation.pivot(index='Dataset', columns='Model', values='range')
range_pivot = range_pivot.reindex(index=datasets,columns=present_models)

ax = range_pivot.plot(kind='bar', figsize=(8, 4), rot=0,color = palette, width=0.85)
#ax.set_title('Range of recall@1 Across Representations (Max - Min)')
#ax.set_xlabel('Dataset')
#ax.set_ylabel('Range')
#ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.get_legend().remove()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# -----------------------------
# 6) Single-number view for your claim
# -----------------------------
print("\nRanking by avg_range_across_models (higher = more variation):")
print(dataset_summary.sort_values('avg_range_across_models', ascending=False).to_string(index=False))
